In [15]:
import os
import warnings
import tensorflow as tf

os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
warnings.filterwarnings('ignore')

print(tf.__version__)

2.18.0


In [2]:
import kagglehub
deepfake_faces_path = kagglehub.dataset_download('dagnelies/deepfake-faces')

In [3]:
import shutil
import pandas as pd
from sklearn.model_selection import train_test_split

In [4]:
# Configuring Paths
dataset_dir = "/kaggle/input/deepfake-faces"
metadata_path = "/kaggle/input/deepfake-faces/metadata.csv"
output_dir = "/kaggle/working/deepfake_faces_split"
image_dir = "/kaggle/input/deepfake-faces/faces_224"

In [5]:
df = pd.read_csv(metadata_path)  # Loading the Metadata

In [6]:
df = df.drop(columns=['original_width', 'original_height', 'original'])
df['videoname'] = df['videoname'].str[:-4] + ".jpg"  # Converting 'videoname' .mp4 -> .jpg
df['label'] = df['label'].str.upper()

print(df.head())
print(f"Number of samples: {len(df)}")

        videoname label
0  aznyksihgl.jpg  FAKE
1  gkwmalrvcj.jpg  FAKE
2  lxnqzocgaq.jpg  FAKE
3  itsbtrrelv.jpg  FAKE
4  ddvgrczjno.jpg  FAKE
Number of samples: 95634


In [18]:
print(df.columns)
print(f"Number of rows: {len(df)}")

Index(['videoname', 'label'], dtype='object')
Number of rows: 95634


In [8]:
print(df['label'].value_counts())  # Checking if both 'REAL' and 'FAKE' exist
print(f"Final cleaned sample count: {len(df)}")

label
FAKE    79341
REAL    16293
Name: count, dtype: int64
Final cleaned sample count: 95634


In [9]:
from sklearn.model_selection import train_test_split

train_val_df, test_df = train_test_split(df, test_size=0.15, stratify=df['label'], random_state=42)  # First splitting into train+val and test
train_df, val_df = train_test_split(train_val_df, test_size=0.15, stratify=train_val_df['label'], random_state=42)  # Now splitting train+val into train and val

print(f"Train: {len(train_df)}, Val: {len(val_df)}, Test: {len(test_df)}")

Train: 69094, Val: 12194, Test: 14346


In [10]:
# Creating output directories
for split in ['train', 'val', 'test']:
    for label in ['REAL', 'FAKE']:
        split_path = os.path.join(output_dir, split, label)
        os.makedirs(split_path, exist_ok=True)

In [12]:
import os
import shutil
from tqdm import tqdm

def copy_files(df, split_name, src_folder=image_dir, dst_root=output_dir):
    """
    Copies image files from src_folder to structured destination folders under dst_root/split_name/label/
    """
    for label in df['label'].unique():
        os.makedirs(os.path.join(dst_root, split_name, label), exist_ok=True)

    for _, row in tqdm(df.iterrows(), total=len(df), desc=f"Copying {split_name} files"):
        src = os.path.join(src_folder, row['videoname'])
        dst = os.path.join(dst_root, split_name, row['label'], row['videoname'])
        if os.path.exists(src):
            shutil.copy(src, dst)

In [13]:
# Copying files
copy_files(train_df, 'train')
copy_files(val_df, 'val')
copy_files(test_df, 'test')

Copying test files: 100%|██████████| 14346/14346 [02:59<00:00, 79.86it/s]


In [14]:
# Generating the metadata_split.csv
combined_df = pd.concat([
    train_df.assign(split='train'),
    val_df.assign(split='val'),
    test_df.assign(split='test')
])

combined_df['path'] = combined_df.apply(lambda row: os.path.join(row['split'], row['label'], row['videoname']), axis=1)
combined_df[['split', 'videoname', 'label', 'path']].to_csv(os.path.join(output_dir, 'metadata_split.csv'), index=False)

print("✅ Dataset split complete. Folder structure and metadata_split.csv created.")

✅ Dataset split complete. Folder structure and metadata_split.csv created.


In [16]:
shutil.make_archive('deepfake_split_dataset', 'zip', '/kaggle/working/deepfake_faces_split')

'/kaggle/working/deepfake_split_dataset.zip'

In [17]:
import os

if os.path.exists('deepfake_split_dataset.zip'):
    print("✅ Zip file created:", os.path.getsize('deepfake_split_dataset.zip') // 1024, "KB")
else:
    print("❌ Failed to create zip file")

✅ Zip file created: 439391 KB
